# Chapter 9: Multiple and Logistic Regression

> **Course:** Statistics | **Instructor:** Haydar Kılıç | **Artificial Intelligence Engineering**

This notebook is a hands-on Python demonstration of the theoretical topics covered in Chapter 9.

## Contents
1. [Multiple Linear Regression](#1)
2. [Categorical Variables and Reference Level](#2)
3. [Interpretation of Regression Coefficients](#3)
4. [R² and Adjusted R²](#4)
5. [Model Selection (Backward Elimination)](#5)
6. [Checking Model Conditions](#6)
7. [Logistic Regression and GLM](#7)
8. [Donner Party: Survival Analysis](#8)
9. [Bird Keeping and Lung Cancer](#9)
10. [ROC Curve and Spam Filter](#10)

In [ ]:
# ── Required Libraries ────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.dpi': 110,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('tab10')

print('Libraries loaded successfully ✓')

---
<a id='1'></a>
## 1. Multiple Linear Regression

**Simple linear regression:** $\hat{y} = \beta_0 + \beta_1 x$

**Multiple linear regression:** $\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$

The **book weights** example from the slides: we will predict a book's weight from its *volume* and *cover type*.

In [ ]:
# ── Book Weights Dataset (Table from Slides) ─────────────────────────────────
book_data = {
    'weight':  [800, 950, 1050, 350,  750,  600, 1075, 250, 700, 650, 975, 350, 950, 425, 725],
    'volume':  [885, 1016, 1125, 239, 701,  641, 1228, 412, 953, 929, 1492, 419, 1010, 595, 1034],
    'cover':   ['hardcover','hardcover','hardcover','hardcover','hardcover',
                'hardcover','hardcover',
                'paperback','paperback','paperback','paperback',
                'paperback','paperback','paperback','paperback']
}
books = pd.DataFrame(book_data)
books['cover_code'] = (books['cover'] == 'paperback').astype(int)  # 0=hardcover, 1=paperback

print(books.to_string(index=False))

In [ ]:
# ── 1a. Simple Regression with Volume Only ───────────────────────────────────
model_simple = smf.ols('weight ~ volume', data=books).fit()
print('=== MODEL 1: Volume Only ===')
print(model_simple.summary().tables[1])
print(f'R² = {model_simple.rsquared:.4f}   Adjusted R² = {model_simple.rsquared_adj:.4f}')

In [ ]:
# ── 1b. Multiple Regression with Volume + Cover Type ─────────────────────────
model_multiple = smf.ols('weight ~ volume + cover_code', data=books).fit()
print('=== MODEL 2: Volume + Cover Type ===')
print(model_multiple.summary().tables[1])
print(f'R² = {model_multiple.rsquared:.4f}   Adjusted R² = {model_multiple.rsquared_adj:.4f}')

**Linear model equations:**

$$\widehat{\text{weight}} = 197.96 + 0.72 \times \text{volume} - 184.05 \times \text{cover\_paperback}$$

- **Hardcover** (`cover_code = 0`): $\hat{y} = 197.96 + 0.72 \times \text{volume}$
- **Paperback** (`cover_code = 1`): $\hat{y} = 13.91 + 0.72 \times \text{volume}$

In [ ]:
# ── 1c. Visualising the Two Regression Lines ─────────────────────────────────
b0      = model_multiple.params['Intercept']
b1      = model_multiple.params['volume']
b_cover = model_multiple.params['cover_code']

vol_range  = np.linspace(200, 1550, 200)
y_hard     = b0 + b1 * vol_range
y_paper    = b0 + b_cover + b1 * vol_range

fig, ax = plt.subplots(figsize=(8, 5))

hard_mask = books['cover'] == 'hardcover'
ax.scatter(books.loc[hard_mask,  'volume'], books.loc[hard_mask,  'weight'],
           marker='s', color='steelblue', s=70, label='Hardcover', zorder=3)
ax.scatter(books.loc[~hard_mask, 'volume'], books.loc[~hard_mask, 'weight'],
           marker='o', color='tomato',     s=70, label='Paperback', zorder=3)

ax.plot(vol_range, y_hard,  color='steelblue', lw=2,
        label=f'Hardcover: ŷ = {b0:.1f} + {b1:.2f}×volume')
ax.plot(vol_range, y_paper, color='tomato', lw=2, linestyle='--',
        label=f'Paperback: ŷ = {b0+b_cover:.1f} + {b1:.2f}×volume')

ax.set_xlabel('Volume (cm³)')
ax.set_ylabel('Weight (g)')
ax.set_title('Book Weights – Multiple Regression Model')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f'\nCoefficient Interpretation:')
print(f'  Volume slope  : Holding cover type constant, a 1 cm³ increase in volume raises weight by ~{b1:.2f} g.')
print(f'  Cover slope   : Holding volume constant, paperbacks are ~{-b_cover:.0f} g lighter than hardcovers.')
print(f'  Intercept     : Estimated weight of a zero-volume hardcover is {b0:.1f} g (meaningless but anchors the line).')

In [ ]:
# ── 1d. Example Prediction ───────────────────────────────────────────────────
# Question from slides: predicted weight of a 600 cm³ paperback book?
vol_example  = 600
pred_paper   = b0 + b_cover + b1 * vol_example
pred_hard    = b0            + b1 * vol_example

print(f'Predictions for a 600 cm³ book:')
print(f'  Paperback : {b0:.2f} + {b1:.2f}×{vol_example} + ({b_cover:.2f})×1 = {pred_paper:.2f} g')
print(f'  Hardcover : {b0:.2f} + {b1:.2f}×{vol_example} + ({b_cover:.2f})×0 = {pred_hard:.2f} g')

---
<a id='2'></a>
## 2. Categorical Variables and Reference Level

Categorical variables are added to the model as **dummy variables**.
- For a categorical variable with $k$ levels, $k-1$ dummy variables are created.
- The level not included in the model is the **reference level**; its coefficient is taken as zero.

Example from the slides: modelling poverty rate with 4 regions (northeast, midwest, west, south).

In [ ]:
# ── Poverty – Region Example (Coefficients from Slides) ──────────────────────
# From slides: reference = northeast, intercept = 9.50
np.random.seed(42)
region_cats   = ['northeast', 'midwest', 'west', 'south']
region_params = {'northeast': 0, 'midwest': 0.03, 'west': 1.79, 'south': 4.16}
intercept     = 9.50

n = 51
regions = np.random.choice(region_cats, size=n)
poverty = [intercept + region_params[r] + np.random.normal(0, 1.5) for r in regions]

df_region = pd.DataFrame({'region': regions, 'poverty': poverty})

# Reference level = northeast (order with CategoricalDtype)
df_region['region'] = pd.Categorical(df_region['region'],
                                      categories=region_cats)

model_region = smf.ols('poverty ~ C(region, Treatment("northeast"))', data=df_region).fit()
print(model_region.summary().tables[1])
print('\n→ Reference level: northeast (embedded in the intercept)')

In [ ]:
# ── Visualisation: Poverty Rate Box Plot by Region ───────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
df_region.boxplot(column='poverty', by='region', ax=ax,
                  boxprops=dict(color='steelblue'),
                  medianprops=dict(color='tomato', lw=2))
ax.set_title('Poverty Rate by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Poverty Rate (%)')
plt.suptitle('')
plt.tight_layout()
plt.show()

---
<a id='3'></a>
## 3. Multicollinearity

> **Multicollinearity** arises when explanatory variables are highly correlated with each other. This destabilises coefficient estimates and makes interpretation more difficult.

In [ ]:
# ── Poverty – Female Head of Household + White Example ───────────────────────
np.random.seed(7)
n = 51
female_head = np.random.uniform(8, 18, n)
# Create negative correlation between 'white' and 'female_head' (r ≈ -0.75)
white   = 95 - 2.0 * female_head + np.random.normal(0, 8, n)
white   = np.clip(white, 30, 100)
poverty = 3.31 + 0.69 * female_head + np.random.normal(0, 1.8, n)

df_pov = pd.DataFrame({'poverty':     poverty,
                        'female_head': female_head,
                        'white':       white})

m1 = smf.ols('poverty ~ female_head',           data=df_pov).fit()
m2 = smf.ols('poverty ~ female_head + white',   data=df_pov).fit()

print('Model 1 – Female head only:')
print(m1.summary().tables[1])
print(f'  R² = {m1.rsquared:.3f}  |  Adj. R² = {m1.rsquared_adj:.3f}\n')

print('Model 2 – Female head + White:')
print(m2.summary().tables[1])
print(f'  R² = {m2.rsquared:.3f}  |  Adj. R² = {m2.rsquared_adj:.3f}')

In [ ]:
# ── VIF (Variance Inflation Factor) to Detect Multicollinearity ──────────────
# VIF > 5-10 → multicollinearity present
X_vif  = sm.add_constant(df_pov[['female_head', 'white']])
vif_data = pd.DataFrame({
    'Variable': ['female_head', 'white'],
    'VIF':      [variance_inflation_factor(X_vif.values, i+1) for i in range(2)]
})
print('Variance Inflation Factors (VIF):')
print(vif_data.to_string(index=False))
print('\nCorrelation matrix:')
print(df_pov[['female_head', 'white']].corr().round(3))
print('\n→ If VIF is high, consider removing one variable from the pair.')

---
<a id='4'></a>
## 4. R² and Adjusted R²

$$R^2 = \frac{SS_{Model}}{SS_{Total}} = 1 - \frac{SS_{Error}}{SS_{Total}}$$

$$R^2_{\text{adj}} = 1 - \frac{SS_{Error}}{SS_{Total}} \times \frac{n-1}{n-p-1}$$

> **Why Adjusted R²?** When a meaningless variable is added to a model, R² always increases but Adjusted R² does not (it penalises).

In [ ]:
# ── R² and Adjusted R² Calculation (Manual) ──────────────────────────────────
y       = df_pov['poverty'].values
y_hat1  = m1.fittedvalues.values
y_hat2  = m2.fittedvalues.values

SS_total = np.sum((y - y.mean())**2)
SS_err1  = np.sum((y - y_hat1)**2)
SS_err2  = np.sum((y - y_hat2)**2)
n_obs    = len(y)

def adj_r2(SS_err, SS_total, n, p):
    return 1 - (SS_err / SS_total) * ((n - 1) / (n - p - 1))

print('=== ANOVA-style Calculation ===')
print(f'SS_Total = {SS_total:.2f}')
print(f'SS_Error (Model 1, p=1) = {SS_err1:.2f}')
print(f'SS_Error (Model 2, p=2) = {SS_err2:.2f}')
print()
print(f'Model 1  R²     = {1 - SS_err1/SS_total:.3f}')
print(f'Model 1  Adj R² = {adj_r2(SS_err1, SS_total, n_obs, 1):.3f}')
print(f'Model 2  R²     = {1 - SS_err2/SS_total:.3f}')
print(f'Model 2  Adj R² = {adj_r2(SS_err2, SS_total, n_obs, 2):.3f}')
print()
print('→ Even though R² increases, Adjusted R² barely changes → white adds no additional information.')

In [ ]:
# ── Visualisation: R² vs Adjusted R² ─────────────────────────────────────────
# How does R² change as random variables are added to the model?
np.random.seed(42)
r2_list, r2_adj_list = [], []
max_p = 15

for p in range(1, max_p + 1):
    X_extra = np.random.randn(n_obs, p)
    X_all   = np.column_stack([df_pov['female_head'].values, X_extra])
    X_sm    = sm.add_constant(X_all)
    m_tmp   = sm.OLS(y, X_sm).fit()
    r2_list.append(m_tmp.rsquared)
    r2_adj_list.append(m_tmp.rsquared_adj)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, max_p+1), r2_list,     'o-', color='steelblue', label='R²')
ax.plot(range(1, max_p+1), r2_adj_list, 's--', color='tomato',    label='Adjusted R²')
ax.set_xlabel('Number of Predictors in the Model')
ax.set_ylabel('R²')
ax.set_title('R² vs Adjusted R² as Random Variables are Added')
ax.legend()
ax.axhline(y=r2_list[0], color='gray', linestyle=':', lw=1)
plt.tight_layout()
plt.show()
print('→ R² always increases; Adjusted R² decreases when a meaningless variable is added.')

---
<a id='5'></a>
## 5. Model Selection – Backward Elimination

We will perform model selection on the **professor evaluations** dataset from the slides.

**Strategy 1 – Adjusted R²:**
1. Start with the full model
2. Remove each variable one at a time, record Adj. R²
3. Select the model that most increases Adj. R²
4. Repeat

**Strategy 2 – p-value:**
1. Start with the full model
2. Remove the variable with the highest p-value
3. Repeat until all remaining variables are significant

In [ ]:
# ── Professor Evaluations Dataset ────────────────────────────────────────────
# Synthetic data close to the openintro evals dataset
np.random.seed(42)
n = 463

beauty       = np.random.normal(0, 0.7, n)
gender       = np.random.choice([0, 1], size=n, p=[0.42, 0.58])  # 0=female, 1=male
age          = np.random.randint(28, 73, n)
formal       = np.random.choice([0, 1], size=n, p=[0.6, 0.4])
lower        = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
non_english  = np.random.choice([0, 1], size=n, p=[0.85, 0.15])  # 0=english, 1=non-english
minority     = np.random.choice([0, 1], size=n, p=[0.85, 0.15])
students     = np.random.randint(8, 120, n)
tenure_track = np.random.choice([0, 1, 2], size=n, p=[0.2, 0.3, 0.5])

# True model: coefficients from slides + noise
eval_score = (4.63
              + 0.108 * beauty
              + 0.204 * gender
              - 0.009 * age
              + 0.151 * formal
              + 0.058 * lower
              - 0.216 * non_english
              - 0.071 * minority
              - 0.0004 * students
              - 0.193 * (tenure_track == 1)
              - 0.157 * (tenure_track == 2)
              + np.random.normal(0, 0.45, n))
eval_score = np.clip(eval_score, 2.0, 5.0)

df_prof = pd.DataFrame({
    'score':        eval_score,
    'beauty':       beauty,
    'gender':       gender,
    'age':          age,
    'formal':       formal,
    'lower':        lower,
    'non_english':  non_english,
    'minority':     minority,
    'students':     students,
    'tenure_tt':    (tenure_track == 1).astype(int),
    'tenure_t':     (tenure_track == 2).astype(int)
})

print('Dataset dimensions:', df_prof.shape)
df_prof.head()

In [ ]:
# ── Full Model ────────────────────────────────────────────────────────────────
full_formula = ('score ~ beauty + gender + age + formal + lower'
                ' + non_english + minority + students + tenure_tt + tenure_t')
full_model = smf.ols(full_formula, data=df_prof).fit()

print('=== FULL MODEL ===')
print(full_model.summary().tables[1])
print(f'\nAdjusted R² = {full_model.rsquared_adj:.4f}')

In [ ]:
# ── Backward Elimination by p-value ──────────────────────────────────────────
def backward_elimination_p(df, response, predictors, threshold=0.05, verbose=True):
    """Backward elimination using the p-value approach."""
    current = list(predictors)
    step    = 0
    while True:
        formula = response + ' ~ ' + ' + '.join(current)
        model   = smf.ols(formula, data=df).fit()
        pvals   = model.pvalues.drop('Intercept')
        max_p   = pvals.max()
        if max_p > threshold:
            to_remove = pvals.idxmax()
            if verbose:
                print(f'Step {step+1}: removed "{to_remove}"  (p={max_p:.4f})  |  Remaining Adj.R²={model.rsquared_adj:.4f}')
            current.remove(to_remove)
            step += 1
        else:
            break
    return model, current

all_predictors = ['beauty','gender','age','formal','lower',
                  'non_english','minority','students','tenure_tt','tenure_t']

print('=== BACKWARD ELIMINATION BY p-VALUE ===')
final_model_p, remaining_p = backward_elimination_p(df_prof, 'score', all_predictors)
print(f'\nFinal model: {" + ".join(remaining_p)}')
print(final_model_p.summary().tables[1])
print(f'Adjusted R² = {final_model_p.rsquared_adj:.4f}')

In [ ]:
# ── Backward Elimination by Adjusted R² ──────────────────────────────────────
def backward_elimination_r2(df, response, predictors, verbose=True):
    """Backward elimination using the Adjusted R² approach."""
    current = list(predictors)
    formula = response + ' ~ ' + ' + '.join(current)
    current_r2 = smf.ols(formula, data=df).fit().rsquared_adj
    step = 0
    while len(current) > 1:
        r2_table = {}
        for var in current:
            candidate = [v for v in current if v != var]
            f = response + ' ~ ' + ' + '.join(candidate)
            r2_table[var] = smf.ols(f, data=df).fit().rsquared_adj
        best_remove = max(r2_table, key=r2_table.get)
        best_r2     = r2_table[best_remove]
        if best_r2 >= current_r2:
            if verbose:
                print(f'Step {step+1}: removed "{best_remove}"  |  New Adj.R²={best_r2:.4f}')
            current.remove(best_remove)
            current_r2 = best_r2
            step += 1
        else:
            break
    formula = response + ' ~ ' + ' + '.join(current)
    return smf.ols(formula, data=df).fit(), current

print('=== BACKWARD ELIMINATION BY ADJUSTED R² ===')
final_model_r2, remaining_r2 = backward_elimination_r2(df_prof, 'score', all_predictors)
print(f'\nFinal model: {" + ".join(remaining_r2)}')
print(f'Adjusted R² = {final_model_r2.rsquared_adj:.4f}')

---
<a id='6'></a>
## 6. Checking Model Conditions

Multiple linear regression rests on 4 conditions:
1. **Normality:** Residuals are approximately normally distributed
2. **Constant variance (homoscedasticity):** Variance of residuals should not vary with fitted values
3. **Independence:** Residuals are independent
4. **Linearity:** Each explanatory variable has a linear relationship with the response

In [ ]:
# ── Diagnostic Plots for the 4 Conditions ────────────────────────────────────
residuals   = final_model_p.resid.values
fitted      = final_model_p.fittedvalues.values
order_index = np.arange(1, len(residuals)+1)

fig, axs = plt.subplots(2, 2, figsize=(11, 8))
fig.suptitle('Graphical Check of Model Conditions', fontsize=14, y=1.01)

# (1) Normality: Histogram of Residuals
axs[0,0].hist(residuals, bins=25, color='steelblue', edgecolor='white', alpha=0.85)
axs[0,0].set_title('(1) Histogram of Residuals\n→ Does it look normal?')
axs[0,0].set_xlabel('Residuals')
axs[0,0].set_ylabel('Frequency')

# (1b) Normal Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axs[0,1])
axs[0,1].set_title('(1b) Normal Probability Plot (Q-Q)\n→ Do points fall on the diagonal?')
axs[0,1].get_lines()[0].set(color='steelblue', markersize=3)

# (2) Constant Variance: Residuals vs Fitted
axs[1,0].scatter(fitted, residuals, alpha=0.5, s=18, color='steelblue')
axs[1,0].axhline(0, color='tomato', lw=1.5, linestyle='--')
axs[1,0].set_title('(2) Residuals vs Fitted Values\n→ Random scatter (constant variance)?')
axs[1,0].set_xlabel('Fitted (Predicted)')
axs[1,0].set_ylabel('Residuals')

# (3) Independence: Residuals vs Collection Order
axs[1,1].scatter(order_index, residuals, alpha=0.4, s=10, color='steelblue')
axs[1,1].axhline(0, color='tomato', lw=1.5, linestyle='--')
axs[1,1].set_title('(3) Residuals vs Observation Order\n→ No trend?')
axs[1,1].set_xlabel('Observation Order')
axs[1,1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

In [ ]:
# ── (4) Linearity: Residuals vs Numerical Predictors ─────────────────────────
numerical_vars = ['beauty', 'age']

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for i, var in enumerate(numerical_vars):
    axs[i].scatter(df_prof[var], residuals, alpha=0.4, s=12, color='steelblue')
    axs[i].axhline(0, color='tomato', lw=1.5, linestyle='--')
    axs[i].set_xlabel(var)
    axs[i].set_ylabel('Residuals')
    axs[i].set_title(f'(4) Residuals vs {var}\n→ Linear relationship?')
plt.suptitle('Linearity Check', fontsize=13)
plt.tight_layout()
plt.show()

---
<a id='7'></a>
## 7. Logistic Regression and GLM

Used for a **binary categorical** response variable (e.g. survived/died, spam/not spam).

**Logit function (link function):**
$$\text{logit}(p) = \log\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

**Back-transforming to get a prediction:**
$$p = \frac{e^{\eta}}{1 + e^{\eta}} = \frac{1}{1 + e^{-\eta}}$$

In [ ]:
# ── Visualising the Logit and Inverse-Logit Functions ────────────────────────
fig, axs = plt.subplots(1, 2, figsize=(11, 4))

# Logit: p → (-∞, +∞)
p_range   = np.linspace(0.001, 0.999, 300)
logit_val = np.log(p_range / (1 - p_range))
axs[0].plot(p_range, logit_val, color='steelblue', lw=2.5)
axs[0].axhline(0, color='gray', lw=0.8, ls=':')
axs[0].axvline(0.5, color='tomato', lw=1, ls='--', label='p=0.5 → logit=0')
axs[0].set_xlabel('p (probability)')
axs[0].set_ylabel('logit(p)')
axs[0].set_title('Logit Function\n[0,1] → (-∞, +∞)')
axs[0].legend()

# Inverse-logit (logistic/sigmoid): η → [0,1]
eta_range = np.linspace(-6, 6, 300)
sigmoid   = 1 / (1 + np.exp(-eta_range))
axs[1].plot(eta_range, sigmoid, color='steelblue', lw=2.5)
axs[1].axhline(0.5, color='tomato', lw=1, ls='--', label='p=0.5')
axs[1].axhline(0.0, color='gray',   lw=0.8, ls=':')
axs[1].axhline(1.0, color='gray',   lw=0.8, ls=':')
axs[1].set_xlabel('η (linear predictor)')
axs[1].set_ylabel('p = σ(η)')
axs[1].set_title('Inverse-Logit (Sigmoid) Function\n(-∞, +∞) → [0,1]')
axs[1].legend()

plt.tight_layout()
plt.show()

print('Odds Ratio Reminder:')
for p in [0.1, 0.5, 0.8, 0.9]:
    odds = p / (1 - p)
    print(f'  p = {p:.1f}  →  odds = {odds:.3f}  →  log-odds = {np.log(odds):.3f}')

---
<a id='8'></a>
## 8. Example – Donner Party: Survival Analysis

The 1846–1847 Sierra Nevada disaster. Dataset of 45 individuals. Response variable: `status` (1=Survived, 0=Died).

In [ ]:
# ── Donner Party Dataset ──────────────────────────────────────────────────────
donner_data = [
    (23,'Male',0),(40,'Female',1),(40,'Male',1),(30,'Male',0),(28,'Male',0),
    (40,'Male',0),(45,'Female',0),(62,'Male',0),(65,'Male',0),(45,'Male',0),
    (25,'Female',1),(28,'Female',1),(28,'Male',0),(23,'Male',1),(22,'Male',0),
    (23,'Male',0),(28,'Male',0),(15,'Male',0),(47,'Male',0),(57,'Male',0),
    (20,'Female',1),(18,'Female',1),(25,'Male',0),(60,'Male',0),(25,'Male',1),
    (35,'Female',1),(32,'Male',1),(24,'Female',1),(23,'Male',0),(28,'Female',1),
    (15,'Male',0),(50,'Male',0),(21,'Female',1),(25,'Male',0),(46,'Male',1),
    (32,'Female',0),(30,'Male',1),(25,'Female',1),(25,'Male',0),(28,'Male',1),
    (32,'Male',0),(20,'Male',0),(23,'Male',1),(24,'Male',0),(25,'Female',1)
]
donner = pd.DataFrame(donner_data, columns=['age', 'sex', 'status'])
donner['sex_code'] = (donner['sex'] == 'Female').astype(int)

print(f'Total: {len(donner)} individuals')
print(donner.groupby(['sex','status']).size().unstack().rename(columns={0:'Died',1:'Survived'}))

In [ ]:
# ── Model 1: Age Only ─────────────────────────────────────────────────────────
model_age = smf.logit('status ~ age', data=donner).fit()
print(model_age.summary())

In [ ]:
# ── Verifying Predictions from the Slides ────────────────────────────────────
b0_age = model_age.params['Intercept']
b1_age = model_age.params['age']

def log_odds(age_val):
    return b0_age + b1_age * age_val

def survival_probability(age_val):
    eta  = log_odds(age_val)
    odds = np.exp(eta)
    p    = odds / (1 + odds)
    return eta, odds, p

print('Calculations from Slides:')
print(f'Coefficients: Intercept={b0_age:.4f}, Age={b1_age:.4f}\n')
for age_test in [0, 25, 50]:
    eta, odds, p = survival_probability(age_test)
    print(f'  Age={age_test:2d}:  log-odds={eta:.4f}  →  odds={odds:.3f}  →  p={p:.3f}')

In [ ]:
# ── Model 2: Age + Sex ────────────────────────────────────────────────────────
model_age_sex = smf.logit('status ~ age + sex_code', data=donner).fit()
print(model_age_sex.summary())

b0    = model_age_sex.params['Intercept']
b_age = model_age_sex.params['age']
b_sex = model_age_sex.params['sex_code']
print(f'\nOdds Ratio Interpretation:')
print(f'  1-year increase in age:   exp({b_age:.4f}) = {np.exp(b_age):.4f}')
print(f'  Female vs Male (age held constant): exp({b_sex:.4f}) = {np.exp(b_sex):.4f}')

In [ ]:
# ── Logistic Curves: Male and Female Models ───────────────────────────────────
age_range = np.linspace(15, 70, 200)

p_male   = 1 / (1 + np.exp(-(b0 + b_age * age_range)))
p_female = 1 / (1 + np.exp(-(b0 + b_sex + b_age * age_range)))

fig, ax = plt.subplots(figsize=(8, 5))

# Data points (with jitter)
for _, row in donner.iterrows():
    jitter = np.random.uniform(-0.02, 0.02)
    color  = 'tomato'    if row['sex'] == 'Female' else 'steelblue'
    marker = 'D'         if row['sex'] == 'Female' else 'o'
    ax.scatter(row['age'], row['status'] + jitter, color=color, marker=marker,
               s=40, alpha=0.7, zorder=3)

ax.plot(age_range, p_male,   color='steelblue', lw=2.5, label='Male')
ax.plot(age_range, p_female, color='tomato',    lw=2.5, label='Female')

ax.set_xlabel('Age')
ax.set_ylabel('Survival Probability')
ax.set_title('Donner Party – Survival Probability (Age + Sex)')
ax.set_yticks([0, 1])
ax.set_yticklabels(['Died (0)', 'Survived (1)'])
ax.legend()
ax.set_ylim(-0.1, 1.1)
plt.tight_layout()
plt.show()

print('Interpretation: Women and younger individuals had a higher probability of survival.')

In [ ]:
# ── Confidence Interval: Age Coefficient ─────────────────────────────────────
CI_log_odds = (
    b_age - 1.96 * model_age_sex.bse['age'],
    b_age + 1.96 * model_age_sex.bse['age']
)
CI_odds = (np.exp(CI_log_odds[0]), np.exp(CI_log_odds[1]))

print('95% Confidence Interval for the Age Coefficient:')
print(f'  Log-odds CI : ({CI_log_odds[0]:.4f}, {CI_log_odds[1]:.4f})')
print(f'  Odds ratio CI: ({CI_odds[0]:.4f}, {CI_odds[1]:.4f})')
print(f'  Interpretation: Each additional year of age changes survival odds by a factor of {CI_odds[0]:.3f}–{CI_odds[1]:.3f}.')

---
<a id='9'></a>
## 9. Example – Bird Keeping and Lung Cancer

Dutch case-control study (1985). Response: lung cancer present/absent.

In [ ]:
# ── Synthetic Data (Based on Slide Coefficients) ─────────────────────────────
np.random.seed(21)
n_bird = 147

sex_b    = np.random.choice([0, 1], n_bird, p=[0.5, 0.5])   # 0=Male, 1=Female
ses_b    = np.random.choice([0, 1], n_bird, p=[0.5, 0.5])   # 0=Low, 1=High
bird_b   = np.random.choice([0, 1], n_bird, p=[0.45, 0.55]) # 0=NoBird, 1=Bird
age_b    = np.random.randint(30, 66, n_bird)
yr_smk_b = np.random.randint(0, 40, n_bird)
cig_b    = np.random.randint(0, 30, n_bird)

# True model: coefficients from slides
eta_b = (-1.937
         + 0.561 * sex_b
         + 0.105 * ses_b
         + 1.363 * bird_b
         - 0.040 * age_b
         + 0.073 * yr_smk_b
         + 0.026 * cig_b)
p_b  = 1 / (1 + np.exp(-eta_b))
lc_b = np.random.binomial(1, p_b)

df_bird = pd.DataFrame({
    'LC': lc_b, 'SE': sex_b, 'SS': ses_b,
    'BK': bird_b, 'AG': age_b, 'YS': yr_smk_b, 'CD': cig_b
})
print(f'Data: {n_bird} individuals | Lung Cancer: {lc_b.sum()} | No Cancer: {(lc_b==0).sum()}')

In [ ]:
# ── Logistic Regression Model ─────────────────────────────────────────────────
model_bird = smf.logit('LC ~ SE + SS + BK + AG + YS + CD', data=df_bird).fit()
print(model_bird.summary())

In [ ]:
# ── Odds Ratio Interpretation ─────────────────────────────────────────────────
coefs = model_bird.params
print('Odds Ratios (exp(β)):')
or_df = pd.DataFrame({
    'Coefficient (β)': coefs.round(4),
    'Odds Ratio':       np.exp(coefs).round(4)
})
print(or_df.to_string())

b_bird_val = coefs['BK']
b_yr_val   = coefs['YS']
print(f'\nKey Interpretations:')
print(f'  Bird-keeping OR: exp({b_bird_val:.4f}) = {np.exp(b_bird_val):.2f}')
print(f'  → Bird keepers have {np.exp(b_bird_val):.2f}x the lung cancer odds compared to non-keepers.')
print(f'  Smoking years (1 yr) OR: exp({b_yr_val:.4f}) = {np.exp(b_yr_val):.3f}')
print(f'  → Each additional smoking year multiplies the risk by {np.exp(b_yr_val):.3f}x.')

In [ ]:
# ── Odds Ratio ≠ Relative Risk ────────────────────────────────────────────────
# Calculation from slides: assuming P(cancer|no bird) = 0.05
or_bird  = np.exp(b_bird_val)  # estimated OR
p_no_bird = 0.05

p_bird_kept = (or_bird * (p_no_bird / (1 - p_no_bird))) / (
               1 + or_bird * (p_no_bird / (1 - p_no_bird)))
relative_risk = p_bird_kept / p_no_bird

print('Odds Ratio vs Relative Risk:')
print(f'  Assumption: P(cancer|no bird) = {p_no_bird}')
print(f'  P(cancer|keeps bird)          = {p_bird_kept:.3f}')
print(f'  Relative Risk (RR)            = {relative_risk:.2f}')
print(f'  Odds Ratio (OR)               = {or_bird:.2f}')
print(f'\n  ⚠  RR={relative_risk:.2f} ≠ OR={or_bird:.2f}')
print(f'  → Logistic regression says "odds ratio {or_bird:.2f}x", NOT "4 times more likely".')

---
<a id='10'></a>
## 10. ROC Curve, AUC, and Spam Filter

### Key Concepts

| True \ Predicted | Positive | Negative |
|---|---|---|
| **Positive** | True Positive (TP) | False Negative (FN) |
| **Negative** | False Positive (FP) | True Negative (TN) |

$$\text{Sensitivity} = \frac{TP}{TP+FN} \qquad \text{Specificity} = \frac{TN}{FP+TN}$$

In [ ]:
# ── Spam Dataset (Simplified) ─────────────────────────────────────────────────
np.random.seed(99)
n_spam     = 3921
spam_true  = np.random.choice([0, 1], n_spam, p=[0.9, 0.1])

# Predicted probabilities from a logistic regression model
p_pred_spam = np.where(
    spam_true == 1,
    np.random.beta(3, 2, n_spam),    # high probability for spam
    np.random.beta(1, 5, n_spam)     # low probability for non-spam
)

print(f'Total emails : {n_spam}')
print(f'Spam         : {spam_true.sum()} ({spam_true.mean()*100:.1f}%)')
print(f'Not spam     : {(spam_true==0).sum()} ({(spam_true==0).mean()*100:.1f}%)')

In [ ]:
# ── Sensitivity / Specificity for Different Thresholds ───────────────────────
thresholds_list = [0.75, 0.625, 0.5, 0.375, 0.25]

print(f'{"Threshold":>10} {"TP":>6} {"TN":>6} {"FP":>6} {"FN":>6} {"Sensitivity":>12} {"Specificity":>12}')
print('-' * 68)
for thr in thresholds_list:
    pred = (p_pred_spam >= thr).astype(int)
    cm   = confusion_matrix(spam_true, pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
    specificity = TN / (FP + TN) if (FP + TN) > 0 else 0
    print(f'{thr:>10.3f} {TP:>6} {TN:>6} {FP:>6} {FN:>6} {sensitivity:>12.3f} {specificity:>12.3f}')

In [ ]:
# ── ROC Curve and AUC ─────────────────────────────────────────────────────────
fpr, tpr, roc_thresholds = roc_curve(spam_true, p_pred_spam)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(fpr, tpr, color='steelblue', lw=2.5, label=f'ROC Curve (AUC = {roc_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier (AUC = 0.5)')

# Mark the thresholds from the slides
for thr_val in [0.75, 0.5, 0.25]:
    idx = np.argmin(np.abs(roc_thresholds - thr_val))
    ax.plot(fpr[idx], tpr[idx], 'ro', markersize=8)
    ax.annotate(f'p={thr_val}', (fpr[idx]+0.02, tpr[idx]-0.04), fontsize=8, color='tomato')

ax.set_xlabel('False Positive Rate (1 – Specificity)')
ax.set_ylabel('True Positive Rate (Sensitivity)')
ax.set_title('ROC Curve – Spam Filter')
ax.legend(loc='lower right')
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
plt.show()

print(f'AUC = {roc_auc:.3f}')
print('→ AUC=1 is a perfect model; AUC=0.5 is random guessing.')
print('→ The ROC curve shows the sensitivity-specificity trade-off for every possible threshold.')

In [ ]:
# ── Optimal Threshold via Utility Function ────────────────────────────────────
# Utility from slides: TP=+1, TN=+1, FP=-50, FN=-5
def utility(y_true, y_pred):
    cm       = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    return TP * 1 + TN * 1 + FP * (-50) + FN * (-5)

threshold_range  = np.linspace(0.01, 0.99, 200)
utility_values   = []

for thr in threshold_range:
    pred_thr = (p_pred_spam >= thr).astype(int)
    utility_values.append(utility(spam_true, pred_thr))

best_idx      = np.argmax(utility_values)
best_threshold = threshold_range[best_idx]
best_utility   = utility_values[best_idx]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_range, utility_values, color='steelblue', lw=2)
ax.axvline(best_threshold, color='tomato', lw=1.5, linestyle='--',
            label=f'Optimal threshold = {best_threshold:.2f}')
ax.scatter([best_threshold], [best_utility], color='tomato', s=80, zorder=5)
ax.set_xlabel('Threshold (p)')
ax.set_ylabel('Utility U(p)')
ax.set_title('Utility Function – Optimal Threshold Selection')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Optimal Threshold : {best_threshold:.3f}')
print(f'Maximum Utility   : {best_utility}')
print(f'Utility at p=0.75 : {utility(spam_true, (p_pred_spam>=0.75).astype(int))}')

In [ ]:
# ── Sensitivity and Specificity Trade-off Visualisation ──────────────────────
sensitivity_list = []
specificity_list = []

for thr in threshold_range:
    pred_thr = (p_pred_spam >= thr).astype(int)
    cm       = confusion_matrix(spam_true, pred_thr)
    TN, FP, FN, TP = cm[0,0], cm[0,1], cm[1,0], cm[1,1]
    sensitivity_list.append(TP / (TP+FN) if (TP+FN)>0 else 0)
    specificity_list.append(TN / (FP+TN) if (FP+TN)>0 else 0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(threshold_range, sensitivity_list, lw=2, color='steelblue', label='Sensitivity')
ax.plot(threshold_range, specificity_list, lw=2, color='tomato',    label='Specificity')
ax.axvline(best_threshold, color='gray', lw=1.5, linestyle='--',
           label=f'Optimal threshold={best_threshold:.2f}')
ax.set_xlabel('Threshold (p)')
ax.set_ylabel('Rate')
ax.set_title('Sensitivity and Specificity Trade-off by Threshold')
ax.legend()
plt.tight_layout()
plt.show()

print('→ As the threshold decreases: Sensitivity increases, Specificity decreases.')
print('→ As the threshold increases: Sensitivity decreases, Specificity increases.')
print('→ The optimal threshold is determined by the utility function.')

---
## 📌 Chapter Summary

| Topic | Key Point |
|---|---|
| **Multiple Regression** | $\hat{y} = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$ |
| **Categorical Variable** | $k-1$ dummy variables; one is the reference level |
| **Coefficient Interpretation** | "Holding everything else constant..." |
| **R² vs Adjusted R²** | Adjusted R² penalises extra variables |
| **Model Selection** | Backward elimination: via p-value or Adjusted R² |
| **Multicollinearity** | VIF > 5–10 → problem; avoid including highly correlated predictors together |
| **Model Conditions** | Normality, Constant Variance, Independence, Linearity |
| **Logistic Regression** | Binary response; logit link function; odds ratio interpretation |
| **ROC / AUC** | Sensitivity-specificity trade-off across all thresholds |
| **OR ≠ Relative Risk** | Makes a big difference when prevalence is low |

---
*Source: OpenIntro Statistics, 4th Edition – Chapter 9 | Haydar Kılıç*